# SQLite projection producer
Windows producer code; configure explicit source roots and a local DPAPI key before rebuild. Default mode is verify. No production receipts are included.


In [ ]:
from __future__ import annotations
if not __debug__:
    raise RuntimeError("projection_optimized_runtime_refused")

import hashlib
import hmac
import json
import os
import re
import sqlite3
import struct
import unicodedata
import uuid
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import unquote

import lancedb
import yaml
import win32crypt

PROJECT = Path(os.environ.get("MQE_ROOT", Path.cwd())).resolve()
if PROJECT.name == "projection":
    PROJECT = PROJECT.parent
WORKFLOW_ROOT = PROJECT
DRIVE_ROOT = Path(WORKFLOW_ROOT.anchor)
PROJECTION_DIR = PROJECT / "projection"
CONFIG_DIR = PROJECT / "config"
DATA_DIR = Path(os.environ.get("MQE_DATA_DIR", PROJECT / "data" / "projection")).resolve()
SCHEMA_PATH = PROJECTION_DIR / "schema.sql"
AUTHORITY_MAP_PATH = CONFIG_DIR / "authority_map.v1.json"
ENTITY_ALIAS_PATH = CONFIG_DIR / "entity_aliases.v1.json"
CURRENT_PATH = DATA_DIR / "CURRENT.json"
MANIFEST_PATH = DATA_DIR / "projection_manifest.v1.json"
PUBLICATION_LOCK_PATH = DATA_DIR / ".projection-publication-lock.sqlite"
MODE = os.environ.get("MQE_MODE", "verify").strip().lower()
assert MODE in {"rebuild", "incremental", "verify", "repair"}

SOURCE_ROOTS_PATH = Path(os.environ.get("MQE_SOURCE_ROOTS_PATH", CONFIG_DIR / "source_roots.v1.json")).resolve()
source_roots = json.loads(SOURCE_ROOTS_PATH.read_text(encoding="utf-8"))
root_keys = {"shared_primary", "cc_active", "phd_hub", "workflow_root", "codex_home", "claude_home", "sport_vault"}
assert set(source_roots) == {"schema_version", "roots"} and source_roots["schema_version"] == "mqe-source-roots/v1"
assert isinstance(source_roots["roots"], dict) and set(source_roots["roots"]) == root_keys
assert all(isinstance(value, str) and value.strip() and "\0" not in value for value in source_roots["roots"].values())
AUTHORITY_ROOTS = {key: (PROJECT / value).resolve() for key, value in source_roots["roots"].items()}
WORKFLOW_ROOT = AUTHORITY_ROOTS["workflow_root"]
if MODE != "verify":
    DATA_DIR.mkdir(parents=True, exist_ok=True)
CODEX_ROOT = AUTHORITY_ROOTS["codex_home"]
CLAUDE_ROOT = AUTHORITY_ROOTS["claude_home"]
MEMORY_ROOTS = {key: AUTHORITY_ROOTS[key] for key in ("shared_primary", "cc_active")}
ENTITY_PATH = AUTHORITY_ROOTS["sport_vault"] / "_wiki" / "entity_to_path.json"
LANCE_PATH = WORKFLOW_ROOT / "openclaw-memory" / "lancedb-pro"
BTI_PATH = WORKFLOW_ROOT / "agent-bridge" / "bti" / "bti_link_registry.sqlite"
BTI_EXCLUDE_PATH = WORKFLOW_ROOT / "agent-bridge" / "bti" / "fail_closed_exclude_narrow.json"
NOW = datetime.now(timezone.utc).replace(microsecond=0)
NOW_ISO = NOW.isoformat().replace("+00:00", "Z")
PUBLICATION_UPDATED_AT = os.environ.get("MQE_PROJECTION_UPDATED_AT") or NOW_ISO
assert re.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?Z", PUBLICATION_UPDATED_AT)
EXPECTED_CURRENT_SHA256 = os.environ.get("MQE_EXPECTED_CURRENT_SHA256")
if EXPECTED_CURRENT_SHA256 is not None:
    assert re.fullmatch(r"[0-9a-f]{64}", EXPECTED_CURRENT_SHA256)
UUID_RE = re.compile(r"^[0-9a-fA-F]{8}(?:-[0-9a-fA-F]{4}){3}-[0-9a-fA-F]{12}$")
SAFE_FIELDS = ("rule", "decision", "summary", "status_line", "description")
SCHEMA_VERSION = "2.0.0"
CHECKSUM_CONTRACT = "logical-tables-v2-real17"
REF_KEY_PATH = Path(os.environ.get("MQE_REF_KEY_PATH", PROJECT / "data" / ".mqe-record-ref-v1.dpapi")).resolve()
REF_ENTROPY = b"mqe-ref-key/v1"
REF_PREFIX = {"record": "rr1", "authority": "ar1", "entity": "er1"}


def canonical_json(value) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_text(value: str) -> str:
    return sha256_bytes(value.encode("utf-8"))


def load_ref_key() -> bytes:
    assert REF_KEY_PATH.is_file(), f"missing DPAPI ref key: {REF_KEY_PATH}"
    protected = REF_KEY_PATH.read_bytes()
    _description, key = win32crypt.CryptUnprotectData(protected, REF_ENTROPY, None, None, 0)
    assert isinstance(key, bytes) and len(key) == 32
    return key


REF_KEY = load_ref_key() if MODE != "verify" else None
REF_KEY_ID = ("rk1_" + hmac.new(REF_KEY, b"mqe-ref-key-id/v1", hashlib.sha256).hexdigest()[:32]) if REF_KEY else None


def make_ref(domain: str, parts: list[str] | tuple[str, ...]) -> str:
    assert REF_KEY is not None and domain in REF_PREFIX and all(isinstance(part, str) for part in parts)
    digest = hmac.new(REF_KEY, digestmod=hashlib.sha256)
    digest.update(b"mqe-ref/v1\0" + domain.encode("utf-8") + b"\0")
    for part in parts:
        raw = part.encode("utf-8")
        digest.update(struct.pack(">I", len(raw)))
        digest.update(raw)
    return f"{REF_PREFIX[domain]}_{digest.hexdigest()}"


def norm(value: str) -> str:
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(value or "")).casefold()).strip()


def loose(value: str) -> str:
    return re.sub(r"[^\w\u3400-\u9fff]+", "", norm(value), flags=re.UNICODE)


def iso_millis(value) -> str:
    try:
        stamp = datetime.fromtimestamp(int(value) / 1000, timezone.utc)
    except Exception:
        stamp = NOW
    return stamp.replace(microsecond=0).isoformat().replace("+00:00", "Z")


def file_iso(path: Path) -> str:
    stamp = datetime.fromtimestamp(path.stat().st_mtime, timezone.utc)
    return stamp.replace(microsecond=0).isoformat().replace("+00:00", "Z")


def frontmatter(text: str) -> tuple[dict, str]:
    if not text.startswith("---"):
        return {}, text
    lines = text.splitlines()
    try:
        end = lines.index("---", 1)
    except ValueError:
        return {}, text
    raw_header = "\n".join(lines[1:end])
    try:
        meta = yaml.safe_load(raw_header) or {}
    except yaml.YAMLError:
        allowed = {*SAFE_FIELDS, "name", "type", "kind", "status", "lance_uuid", "lancedb_id", "lancedb_text_sha256", "supersedes", "superseded_by", "project_id", "entity", "scope", "importance", "answerability", "authority", "bti_anchor", "valid_from", "valid_to"}
        meta = {"_parse_status": "scalar_fallback"}
        for line in lines[1:end]:
            match = re.match(r"^([A-Za-z_][A-Za-z0-9_-]*):\s*(.*)$", line)
            if not match or match.group(1) not in allowed:
                continue
            value = match.group(2).strip()
            if len(value) >= 2 and value[0] == value[-1] and value[0] in {chr(34), chr(39)}:
                value = value[1:-1]
            meta[match.group(1)] = value
    return (meta if isinstance(meta, dict) else {}), "\n".join(lines[end + 1 :]).strip()


_fallback_fixture = frontmatter("---\ndescription: preferred browser ladder: /ll trigger\ntype: reference\n---\nbody")[0]
assert _fallback_fixture["description"] == "preferred browser ladder: /ll trigger"
assert _fallback_fixture["_parse_status"] == "scalar_fallback"


def contained_memory_location(candidate: Path) -> tuple[str, str] | None:
    matches = []
    for root_id, root in MEMORY_ROOTS.items():
        base = root.resolve()
        try:
            relative = candidate.relative_to(base).as_posix()
        except ValueError:
            continue
        matches.append((len(base.parts), root_id, relative))
    if not matches:
        return None
    _depth, root_id, relative = max(matches, key=lambda item: (item[0], item[1]))
    return root_id, relative


def indexed_memory_entries(index_root_id: str, root: Path) -> tuple[list[dict], dict]:
    assert index_root_id in MEMORY_ROOTS and root.resolve() == MEMORY_ROOTS[index_root_id].resolve(), f"invalid memory index root: {index_root_id}"
    index = root / "MEMORY.md"
    assert index.is_file(), f"missing memory index: {index_root_id}"
    text = index.read_text(encoding="utf-8", errors="strict")
    refs = [(match.group(2), match.group(1)) for match in re.finditer(r"\[([^\]]*)\]\(([^)#?]+\.md)(?:#[^)]*)?\)", text, flags=re.IGNORECASE)]
    refs += [(match.group(1), match.group(2) or Path(match.group(1)).stem) for match in re.finditer(r"\[\[([^]|#?]+\.md)(?:#[^]|]*)?(?:\|([^\]]*))?\]\]", text, flags=re.IGNORECASE)]
    base = root.resolve()
    entries = {}
    audit = {"contract": "configured_memory_roots_v1", "index_root_id": index_root_id, "references_seen": len(refs), "accepted_refs": 0, "accepted_unique_files": 0, "cross_root_refs": 0, "duplicate_refs": 0, "skipped_refs": 0, "events": []}
    for raw_ref, label in refs:
        ref = unquote(raw_ref.strip().replace("\\", "/"))
        ref_sha256 = "sha256:" + sha256_text(ref)
        try:
            candidate = (base / ref).resolve()
        except (OSError, RuntimeError):
            audit["skipped_refs"] += 1
            audit["events"].append({"reason": "invalid_path", "index_root_id": index_root_id, "ref_sha256": ref_sha256})
            continue
        location = contained_memory_location(candidate)
        if location is None:
            audit["skipped_refs"] += 1
            audit["events"].append({"reason": "outside_configured_roots", "index_root_id": index_root_id, "ref_sha256": ref_sha256})
            continue
        owner_root_id, relative = location
        logical = {"index_root_id": index_root_id, "owner_root_id": owner_root_id, "relative_path": relative, "ref_sha256": ref_sha256}
        if candidate.name.casefold() in {"memory.md", "active.md"}:
            audit["skipped_refs"] += 1
            audit["events"].append({"reason": "reserved_index_file", **logical})
            continue
        if not candidate.is_file():
            audit["skipped_refs"] += 1
            audit["events"].append({"reason": "missing_target", **logical})
            continue
        audit["accepted_refs"] += 1
        key = (owner_root_id, relative)
        if key in entries:
            audit["duplicate_refs"] += 1
        entry = entries.setdefault(key, {"owner_root_id": owner_root_id, "relative_path": relative, "path": candidate, "labels": set()})
        clean_label = re.sub(r"\s+", " ", str(label or "")).strip()
        if clean_label:
            entry["labels"].add(clean_label)
        if owner_root_id != index_root_id:
            audit["cross_root_refs"] += 1
            audit["events"].append({"reason": "accepted_cross_root", **logical})
    assert entries, f"memory index has no accepted entries: {index_root_id}"
    audit["accepted_unique_files"] = len(entries)
    audit["events"].sort(key=canonical_json)
    return [{**entries[key], "labels": sorted(entries[key]["labels"], key=str.casefold)} for key in sorted(entries)] , audit


def safe_excerpt(meta: dict) -> tuple[str, str]:
    selected = next((meta.get(key).strip() for key in SAFE_FIELDS if isinstance(meta.get(key), str) and meta.get(key).strip()), "")
    if not selected:
        return "", "rejected_unstructured"
    selected = unicodedata.normalize("NFKC", selected)
    if re.fullmatch('^LanceDB\\s+\\S+\\s+的本地轻量伴随卡\\s*\\(\\s*[0-9a-f]{8}\\s*\\)$', selected.strip(), flags=re.IGNORECASE):
        return "", "rejected_generic"
    lowered = selected.casefold()
    if any(token in lowered for token in ("[ai-relay]", "ignore previous", "system:", "assistant:", "<script", "</")):
        return "", "rejected_instruction_like"
    if re.search(r"(?i)(?<![a-z0-9])(?:[a-z]:[\\/]|\\\\[^\\/\s]+[\\/])", selected):
        return "", "rejected_absolute_path"
    if chr(96) * 3 in selected or "\x00" in selected or selected.lstrip().startswith(("/co", "/cl")):
        return "", "rejected_control_surface"
    selected = re.sub(r"<[^>]+>", " ", selected)
    selected = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", selected)
    selected = re.sub(r"\s+", " ", selected).strip()
    return (selected[:360], "accepted") if selected else ("", "rejected_empty")


def alias_list(meta: dict, path: Path, index_aliases=()) -> list[dict]:
    values = [(path.stem.replace("_", " ").replace("-", " "), "filename", 0.92)]
    for key, confidence in (("name", 1.0), ("entity", 1.0), ("project_id", 0.98)):
        value = meta.get(key)
        if isinstance(value, str):
            values.append((value, f"frontmatter:{key}", confidence))
    values.extend((value, "memory_index_label", 0.98) for value in index_aliases if isinstance(value, str))
    aliases = meta.get("aliases", [])
    aliases = [aliases] if isinstance(aliases, str) else aliases if isinstance(aliases, list) else []
    values.extend((value, "frontmatter:aliases", 1.0) for value in aliases if isinstance(value, str))
    output = {}
    for alias, source, confidence in values:
        alias = alias.strip()
        alias_norm = norm(alias)
        if 1 <= len(alias_norm) <= 160:
            key = (alias_norm, alias)
            if key not in output or confidence > output[key]["confidence"]:
                output[key] = {"alias_norm": alias_norm, "alias": alias, "source": source, "confidence": confidence}
    return [output[key] for key in sorted(output)]


def project_id(meta: dict, path: Path) -> str | None:
    value = str(meta.get("project_id") or meta.get("name") or path.stem)
    for prefix in ("project_", "project-"):
        if value.startswith(prefix):
            value = value[len(prefix) :]
            break
    if meta.get("project_id") or str(meta.get("type") or "") == "project" or path.stem.startswith(("project_", "project-")):
        return norm(value).replace("_", "-").replace(" ", "-")
    return None


def string_list(value) -> list[str]:
    if isinstance(value, str):
        return [value] if value.strip() else []
    return [str(item) for item in value if str(item).strip()] if isinstance(value, list) else []


def explicit_lance_id(meta: dict) -> str:
    candidates = []
    for key in ("lance_uuid", "lancedb_id"):
        value = str(meta.get(key) or "").strip().strip(chr(34)).strip(chr(39)).lower()
        if UUID_RE.fullmatch(value):
            candidates.append(value)
    unique = sorted(set(candidates))
    assert len(unique) <= 1, f"conflicting_explicit_lance_ids count={len(unique)}"
    return unique[0] if unique else ""


def local_lance_binding(meta: dict, lance_bindings: dict) -> tuple[str, str]:
    direct = explicit_lance_id(meta)
    if not direct:
        return "", "unbound"
    current = lance_bindings.get(direct)
    if not current:
        return direct, "unknown_id"
    if not current["visible"]:
        return direct, "tombstoned"
    claimed = str(meta.get("lancedb_text_sha256") or "").strip().lower()
    if not re.fullmatch(r"[0-9a-f]{64}", claimed):
        return direct, "missing_or_invalid_hash"
    if claimed != current["text_sha256"]:
        return direct, "hash_mismatch"
    return direct, "matched"


def resolve_record_ref(value, records: dict) -> str:
    text = str(value or "").strip()
    if text in records:
        return text
    match = re.match(r"^(?:LanceDB:)?([0-9a-fA-F]{8}(?:-[0-9a-fA-F]{4}){3}-[0-9a-fA-F]{12}|[0-9a-fA-F]{8})(?:\b|$)", text, flags=re.IGNORECASE)
    assert match, f"invalid supersession reference: {text}"
    token = match.group(1).lower()
    if UUID_RE.fullmatch(token):
        assert token in records, f"dangling supersession reference: {token}"
        return token
    matches = [record_id for record_id in records if record_id.lower().startswith(token)]
    assert len(matches) == 1, f"ambiguous supersession prefix: {token} -> {matches}"
    return matches[0]


def classify_record_ref(value, records: dict, lance_bindings: dict) -> tuple[str | None, str]:
    # Lineage lookup that never asserts: an unresolvable or forgotten link is skipped and counted,
    # so one bad link cannot stop the whole projection from publishing.
    text = str(value or "").strip()
    if text in records:
        return text, "resolved"
    match = re.match(r"^(?:LanceDB:)?([0-9a-fA-F]{8}(?:-[0-9a-fA-F]{4}){3}-[0-9a-fA-F]{12}|[0-9a-fA-F]{8})(?:\b|$)", text, flags=re.IGNORECASE)
    if not match:
        return None, "invalid"
    token = match.group(1).lower()
    # Uniqueness is judged over every known identity, visible or forgotten, so a link whose target was
    # forgotten is never re-pointed at another record that happens to share its short prefix.
    visible = {record_id.lower(): record_id for record_id in records}
    known = set(visible) | set(lance_bindings)
    matches = sorted(identity for identity in known if (identity == token if UUID_RE.fullmatch(token) else identity.startswith(token)))
    if len(matches) > 1:
        return None, "ambiguous"
    if not matches:
        return None, "target_unknown"
    if matches[0] in visible:
        return visible[matches[0]], "resolved"
    binding = lance_bindings.get(matches[0])
    return None, "target_tombstoned" if binding and not binding["visible"] else "target_unknown"


def lineage_reaches(successors: dict, start: str, goal: str) -> bool:
    seen, stack = set(), [start]
    while stack:
        node = stack.pop()
        if node == goal:
            return True
        if node not in seen:
            seen.add(node)
            stack.extend(successors.get(node, ()))
    return False


def logical_pointer(raw: str) -> str | None:
    try:
        path = Path(raw.replace("\\", "/")).resolve()
    except OSError:
        return None
    roots = (
        ("workflow_root", WORKFLOW_ROOT),
        ("sport_vault", AUTHORITY_ROOTS["sport_vault"]),
        ("phd_hub", AUTHORITY_ROOTS["phd_hub"]),
    )
    for root_id, root in roots:
        try:
            return f"{root_id}:{path.relative_to(root.resolve()).as_posix()}"
        except ValueError:
            pass
    return None


def load_bti_overlay() -> tuple[dict, dict, set, dict]:
    exclusions = json.loads(BTI_EXCLUDE_PATH.read_text(encoding="utf-8"))
    assert isinstance(exclusions, (dict, list))
    connection = sqlite3.connect(f"file:{BTI_PATH.as_posix()}?mode=ro", uri=True)
    connection.row_factory = sqlite3.Row
    local_nodes = {}
    for row in connection.execute("SELECT id,metadata_json FROM nodes WHERE type='local_memory'"):
        metadata = json.loads(row["metadata_json"] or "{}")
        root_id, filename = metadata.get("root_id"), metadata.get("filename")
        if root_id in MEMORY_ROOTS and filename:
            local_nodes[(root_id, filename)] = row["id"]
    lance_nodes = {row["id"].removeprefix("lance:") for row in connection.execute("SELECT id FROM nodes WHERE type='lance_memory'")}
    mirror_by_local = {}
    for row in connection.execute("SELECT src_id,dst_id,confidence FROM edges WHERE type='mirrors'"):
        if row["src_id"].startswith("local_memory:") and row["dst_id"].startswith("lance:") and float(row["confidence"]) >= 1.0:
            mirror_by_local[row["src_id"]] = row["dst_id"].removeprefix("lance:")
    connection.close()
    stats = {"local_nodes": len(local_nodes), "lance_nodes": len(lance_nodes), "mirror_edges": len(mirror_by_local), "exclude_entries": len(exclusions)}
    return local_nodes, mirror_by_local, lance_nodes, stats


def lance_projection_visibility(row: dict) -> tuple[bool, str]:
    raw_metadata = row.get("metadata")
    if not isinstance(raw_metadata, str) or not raw_metadata.strip():
        return True, "legacy_visible"
    try:
        metadata = json.loads(raw_metadata)
    except json.JSONDecodeError:
        return True, "legacy_visible"
    if not isinstance(metadata, dict) or "_memory_write_plane" not in metadata:
        return True, "legacy_visible"
    commit = metadata["_memory_write_plane"]
    assert isinstance(commit, dict), "malformed write-plane commit"
    event = commit.get("event")
    assert isinstance(event, dict), "malformed write-plane mutation event"
    schema = event.get("schema")
    if schema == "memory-mutation-event/v1":
        assert event.get("operation") == "save" and event.get("effect") == "stored", "invalid v1 mutation event"
        effect = "stored"
    else:
        assert schema == "memory-mutation-event/v2", "unknown mutation event schema"
        expected = {"schema", "event_id", "mutation_id", "sequence", "operation", "effect", "memory_id", "occurred_at", "request_digest", "companion", "before", "after"}
        assert set(event) == expected, "unexpected v2 mutation event fields"
        operation, effect = event.get("operation"), event.get("effect")
        assert (operation, effect) in {("save", "stored"), ("update", "updated"), ("forget", "tombstoned")}, "invalid v2 mutation transition"
        assert event.get("companion") in {"required", "no_op"}, "invalid v2 mutation companion"
        before, after = event.get("before"), event.get("after")
        if operation == "save":
            assert before is None and isinstance(after, dict), "invalid save snapshot"
        elif operation == "update":
            assert isinstance(before, dict) and isinstance(after, dict), "invalid update snapshot"
        else:
            assert isinstance(before, dict) and after is None, "invalid forget snapshot"
    assert str(event.get("memory_id") or "") == str(row.get("id") or ""), "mutation event memory id mismatch"
    return effect != "tombstoned", effect


# Durable supersession lineage (reserved key of contract memory-supersession/1).
# The supported write plane writes this key only on a correcting save and copies it verbatim into later update/forget
# commits, so a forgotten successor still retires its predecessor. The builder cannot see the creator WAL. It trusts a key
# only on a row whose own supported commit (memory-write-plane-commit/v2 carrying a memory-mutation-event/v2 event)
# vouches for it, with the strict closed shape and creator.memory_id == event memory_id == row id; authenticating the
# creator against its WAL stays the host's write-time duty. A key on a row without a supported commit is skipped and
# counted: a stateless build cannot recover tampering that removed the commit itself.
SUPERSESSION_KEY = "_memory_supersession_v1"
SUPERSESSION_SCHEMA_VERSION = "memory-supersession/1"
SUPERSESSION_SHAPE = {
    "key": frozenset({"schema_version", "predecessor", "creator"}),
    "predecessor": frozenset({"memory_id", "text_sha256", "scope", "category", "revision"}),
    "creator": frozenset({"memory_id", "mutation_id", "idempotency_hash", "request_digest"}),
}
SUPERSESSION_UUID_RE = re.compile(r"[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}")
SUPERSESSION_HEX64_RE = re.compile(r"[0-9a-f]{64}")
SUPERSESSION_SCOPE_RE = re.compile(r"[A-Za-z0-9_.:-]{1,128}")
SUPERSESSION_CATEGORIES = frozenset({"preference", "fact", "decision", "entity", "other"})
SUPERSESSION_DELETED_SCOPE = "__memory_deleted__"
SUPERSESSION_SUPPORTED_COMMIT = ("memory-write-plane-commit/v2", "memory-mutation-event/v2")


class SupersessionLineageStop(AssertionError):
    # Stops publication so the served CURRENT keeps serving. The message carries a fixed code and counts only.
    pass


def parse_supersession_key(value) -> dict | None:
    def matches(pattern, item) -> bool:
        return isinstance(item, str) and pattern.fullmatch(item) is not None

    if not isinstance(value, dict) or set(value) != SUPERSESSION_SHAPE["key"] or value["schema_version"] != SUPERSESSION_SCHEMA_VERSION:
        return None
    predecessor, creator = value["predecessor"], value["creator"]
    if not (isinstance(predecessor, dict) and set(predecessor) == SUPERSESSION_SHAPE["predecessor"] and isinstance(creator, dict) and set(creator) == SUPERSESSION_SHAPE["creator"]):
        return None
    if not (matches(SUPERSESSION_UUID_RE, predecessor["memory_id"]) and matches(SUPERSESSION_HEX64_RE, predecessor["text_sha256"])
            and matches(SUPERSESSION_SCOPE_RE, predecessor["scope"]) and predecessor["scope"] != SUPERSESSION_DELETED_SCOPE
            and isinstance(predecessor["category"], str) and predecessor["category"] in SUPERSESSION_CATEGORIES
            and (predecessor["revision"] is None or matches(SUPERSESSION_UUID_RE, predecessor["revision"]))):
        return None
    if not all(matches(pattern, creator[field]) for field, pattern in (("memory_id", SUPERSESSION_UUID_RE), ("mutation_id", SUPERSESSION_UUID_RE), ("idempotency_hash", SUPERSESSION_HEX64_RE), ("request_digest", SUPERSESSION_HEX64_RE))):
        return None
    return {"predecessor": dict(predecessor), "creator": dict(creator)}


def is_recognizable_supersession_key(value) -> bool:
    # The frozen memory-supersession/1 structure (closed key, predecessor and creator field sets); values are not judged.
    return (isinstance(value, dict) and set(value) == SUPERSESSION_SHAPE["key"] and value["schema_version"] == SUPERSESSION_SCHEMA_VERSION
            and isinstance(value["predecessor"], dict) and set(value["predecessor"]) == SUPERSESSION_SHAPE["predecessor"]
            and isinstance(value["creator"], dict) and set(value["creator"]) == SUPERSESSION_SHAPE["creator"])


SUPERSESSION_INSTANT_RE = re.compile(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}Z")
SUPERSESSION_EVENT_UUID_RE = re.compile(r"[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}", re.IGNORECASE)
SUPERSESSION_EVENT_CONSUMERS = ["projection-invalidation", "downstream-pending"]
SUPERSESSION_TRANSITIONS = (("save", "stored", True, False), ("update", "updated", False, False), ("forget", "tombstoned", False, True))


def is_canonical_v2_commit(commit: dict) -> bool:
    # Mirror of the host canonical parse of an embedded v2 commit (lancedb-memory-host src/mutation-write-plane.ts:
    # parseEmbeddedWritePlane 565-595, validateEvent 538-563, validateSnapshot 529-536). Timestamps take the toISOString
    # form the host writes (660, 607/978), a strict subset of its Date.parse test. No WAL or creator authentication.
    def text(pattern, value) -> bool:
        return isinstance(value, str) and pattern.fullmatch(value) is not None

    def instant(value) -> bool:
        try:
            return text(SUPERSESSION_INSTANT_RE, value) and datetime.strptime(value, "%Y-%m-%dT%H:%M:%S.%fZ") is not None
        except ValueError:
            return False

    def snapshot(value) -> bool:
        return value is None or (isinstance(value, dict) and set(value) == {"category", "scope", "importance", "text_sha256"}
                                 and isinstance(value["category"], str) and value["category"] in SUPERSESSION_CATEGORIES
                                 and text(SUPERSESSION_SCOPE_RE, value["scope"]) and type(value["importance"]) in (int, float)
                                 and 0 <= value["importance"] <= 1 and text(SUPERSESSION_HEX64_RE, value["text_sha256"]))

    event, delivery = commit.get("event"), commit.get("delivery")
    if not (isinstance(event, dict) and isinstance(delivery, dict)):
        return False
    status = delivery.get("status")
    return (set(commit) == {"schema", "idempotency_hash", "request_digest", "event", "delivery"}
            and text(SUPERSESSION_HEX64_RE, commit["idempotency_hash"]) and text(SUPERSESSION_HEX64_RE, commit["request_digest"])
            and set(event) == {"schema", "event_id", "mutation_id", "sequence", "operation", "effect", "memory_id", "occurred_at", "request_digest", "companion", "before", "after"}
            and event["schema"] == "memory-mutation-event/v2"
            and all(text(SUPERSESSION_EVENT_UUID_RE, event[field]) for field in ("event_id", "mutation_id", "memory_id"))
            and type(event["sequence"]) in (int, float) and 1 <= event["sequence"] <= 9007199254740991 and float(event["sequence"]).is_integer()
            and instant(event["occurred_at"])
            and text(SUPERSESSION_HEX64_RE, event["request_digest"]) and event["companion"] in ("required", "no_op")
            and snapshot(event["before"]) and snapshot(event["after"])
            and (event["operation"], event["effect"], event["before"] is None, event["after"] is None) in SUPERSESSION_TRANSITIONS
            and commit["request_digest"] == event["request_digest"]
            and set(delivery) == ({"status", "consumers", "delivered_at"} if status == "delivered" else {"status", "consumers"})
            and delivery["consumers"] == SUPERSESSION_EVENT_CONSUMERS and status in ("pending", "delivered")
            and (status != "delivered" or instant(delivery["delivered_at"])))


def reject_json_constant(token):
    # JSON has no NaN/Infinity/-Infinity literals and the host JSON.parse rejects them; Python json would accept them.
    raise ValueError("non-standard JSON constant")


def classify_supersession_row(row: dict) -> tuple[str, dict | None]:
    # none | untrusted (no supported commit vouches for the key) | malformed | identity_mismatch | trusted
    raw_metadata = row.get("metadata")
    try:
        metadata = json.loads(raw_metadata, parse_constant=reject_json_constant) if isinstance(raw_metadata, str) and raw_metadata.strip() else None
    except ValueError:  # json.JSONDecodeError, or a NaN/Infinity/-Infinity literal rejected by reject_json_constant
        if SUPERSESSION_KEY in raw_metadata:
            return "unreadable_metadata", None  # corrupt bytes cannot prove the reserved key absent: fail closed
        metadata = None  # such a row stays legacy_visible; no key can be read from it
    if not isinstance(metadata, dict) or SUPERSESSION_KEY not in metadata:
        return "none", None
    if "_memory_write_plane" not in metadata:
        return "untrusted", None
    commit = metadata["_memory_write_plane"]
    event = commit.get("event") if isinstance(commit, dict) else None
    if isinstance(event, dict) and (commit.get("schema"), event.get("schema")) == ("memory-write-plane-commit/v1", "memory-mutation-event/v1"):
        if is_recognizable_supersession_key(metadata[SUPERSESSION_KEY]):
            return "v1_commit_with_s2_key", None  # S2 keys are written only with v2 commits: never a legacy pair
        return "untrusted", None
    if not isinstance(event, dict) or (commit.get("schema"), event.get("schema")) != SUPERSESSION_SUPPORTED_COMMIT:
        return "malformed_commit", None
    if not is_canonical_v2_commit(commit):
        return "noncanonical_commit", None  # the host's canonical v2 parse rejects this commit
    lineage = parse_supersession_key(metadata[SUPERSESSION_KEY])
    if lineage is None:
        return "malformed", None
    row_id = str(row.get("id") or "").strip()
    if lineage["creator"]["memory_id"] != row_id or str(event.get("memory_id") or "") != row_id:
        return "identity_mismatch", None
    if event["operation"] == "save" and (lineage["creator"]["mutation_id"], lineage["creator"]["idempotency_hash"], lineage["creator"]["request_digest"]) != (event["mutation_id"], commit["idempotency_hash"], commit["request_digest"]):
        return "creator_commit_mismatch", None  # host: supersession-contract.ts committedSupersessionLineage save binding
    return "trusted", {"successor_id": row_id, **lineage}


def merge_supersession_lineage(records: dict, rows: list[dict], lance_bindings: dict, lineage: dict) -> set:
    # Key edges join the successor's `supersedes`, so the resolution, cycle and multi-successor rules in compile_records
    # treat them exactly like card edges. A committed key that cannot be read, or that is bound to another row, may be the
    # only durable record that a predecessor was retired: skipping it could make that predecessor current again, so
    # publication stops instead (fail closed) and the served CURRENT stays.
    claims, stops = [], defaultdict(int)
    for row in rows:
        outcome, claim = classify_supersession_row(row)
        if outcome == "trusted":
            claims.append(claim)
        elif outcome == "untrusted":
            lineage["s2_skipped_untrusted_source"] += 1
        elif outcome != "none":
            stops[outcome] += 1
    if stops:
        raise SupersessionLineageStop("s2_lineage_stop " + " ".join(f"{outcome}={count}" for outcome, count in sorted(stops.items())))
    claimed = defaultdict(set)
    for claim in claims:
        successor_id, predecessor_id = claim["successor_id"], claim["predecessor"]["memory_id"]
        binding = lance_bindings.get(predecessor_id)
        lineage["s2_trusted_keys"] += 1
        if predecessor_id == successor_id:
            lineage["s2_skipped_self"] += 1
        elif binding is None or (binding["visible"] and predecessor_id not in records):
            lineage["s2_skipped_predecessor_unknown"] += 1  # identity is the full UUID only, never a prefix
        elif not binding["visible"]:
            lineage["s2_skipped_predecessor_tombstoned"] += 1
        elif successor_id in records:
            claimed[predecessor_id].add(successor_id)
            declared = records[successor_id]["supersedes"]
            card_targets = {target for target, kind in (classify_record_ref(value, records, lance_bindings) for value in declared) if kind == "resolved"}
            if card_targets and predecessor_id not in card_targets:
                lineage["s2_card_key_conflict"] += 1  # card and key disagree: both edges hold
            declared.append(predecessor_id)
        else:
            # The successor was forgotten (its tombstone keeps the key) or is not compiled: the retirement is a durable
            # fact of the primary store, so the predecessor stays non-current without an edge to an unpublished record.
            claimed[predecessor_id].add(successor_id)
            records[predecessor_id]["status"] = "superseded"
            state = (lance_bindings.get(successor_id.lower()) or {}).get("state")
            lineage["s2_superseded_by_" + ("tombstoned" if state == "tombstoned" else "unpublished") + "_successor"] += 1
    for successor_ids in claimed.values():
        if len(successor_ids) > 1:
            lineage["s2_conflicting_successors"] += 1  # the host admits one successor per predecessor
    return {predecessor_id for predecessor_id, successor_ids in claimed.items() if len(successor_ids) > 1}  # no winner for these


def compile_records() -> tuple[list[dict], dict]:
    local_nodes, _mirror_by_local, lance_nodes, bti_stats = load_bti_overlay()
    table = lancedb.connect(str(LANCE_PATH)).open_table("memories")
    all_lance_rows = table.to_arrow().to_pylist()
    lance_visibility = defaultdict(int)
    lance_bindings = {}
    lance_rows = []
    for row in all_lance_rows:
        record_id = str(row.get("id") or "").strip().lower()
        text = str(row.get("text") or "")
        visible, state = lance_projection_visibility(row)
        lance_visibility[state] += 1
        if record_id:
            lance_bindings[record_id] = {"visible": visible, "state": state, "text_sha256": sha256_text(text)}
        if visible:
            lance_rows.append(row)
    records = {}
    normalized = defaultdict(list)
    for row in lance_rows:
        record_id = str(row.get("id") or "").strip()
        text = str(row.get("text") or "")
        kind = str(row.get("category") or "other").lower()
        if not record_id or kind not in {"decision", "fact", "preference", "entity", "other"}:
            continue
        content_hash = "sha256:" + sha256_text(text)
        records[record_id] = {
            "record_id": record_id, "origin": "lancedb", "kind": kind, "project_id": None, "entity": None,
            "authority": "memory", "authority_kind": "memory", "authority_rank": 10, "authority_resolution": "resolved",
            "status": "active", "valid_from": None, "valid_to": None, "recorded_time": iso_millis(row.get("timestamp")),
            "supersedes": [], "superseded_by": None, "superseded_by_ref": None, "canonical_record_id": record_id, "content_hash": content_hash,
            "answerability": "memory_allowed", "safe_excerpt": "", "safe_excerpt_status": "rejected_unstructured",
            "source_pointer": f"lancedb:memories:{record_id}", "bti_anchor": f"lance:{record_id}" if record_id in lance_nodes else None, "authority_verified_at": None,
            "fresh_until": None, "search_text": text, "importance": float(row.get("importance") or 0.5),
            "scope": str(row.get("scope") or "global"), "active": 1, "compiled_at": iso_millis(row.get("timestamp")),
            "aliases": [], "sources": [{"root_id": "lancedb", "relative_path": f"memories/{record_id}", "source_hash": content_hash, "merge_basis": "native_source"}],
        }
        normalized[sha256_text(norm(text))].append(record_id)

    indexed_counts = {root_id: 0 for root_id in MEMORY_ROOTS}
    index_audits = {}
    curated_entries = {}
    cross_index_duplicate_refs = 0
    for index_root_id, root in MEMORY_ROOTS.items():
        entries, audit = indexed_memory_entries(index_root_id, root)
        index_audits[index_root_id] = audit
        for entry in entries:
            key = (entry["owner_root_id"], entry["relative_path"])
            cross_index_duplicate_refs += int(key in curated_entries)
            aggregate = curated_entries.setdefault(key, {"path": entry["path"], "labels": set(), "index_roots": set()})
            aggregate["labels"].update(entry["labels"])
            aggregate["index_roots"].add(index_root_id)
    for root_id, _relative_path in curated_entries:
        indexed_counts[root_id] += 1
    index_events = sorted([event for audit in index_audits.values() for event in audit["events"]], key=canonical_json)
    index_ref_audit = {
        "contract": "configured_memory_roots_v1", "references_seen": sum(item["references_seen"] for item in index_audits.values()),
        "accepted_refs": sum(item["accepted_refs"] for item in index_audits.values()), "accepted_unique_files": len(curated_entries),
        "cross_root_refs": sum(item["cross_root_refs"] for item in index_audits.values()), "cross_index_duplicate_refs": cross_index_duplicate_refs,
        "duplicate_refs": cross_index_duplicate_refs + sum(item["duplicate_refs"] for item in index_audits.values()),
        "skipped_refs": sum(item["skipped_refs"] for item in index_audits.values()), "events": index_events,
        "events_sha256": "sha256:" + sha256_text(canonical_json(index_events)),
    }
    merged = 0
    frontmatter_fallbacks = 0
    local_binding_skips = defaultdict(int)
    lineage = defaultdict(int)
    for root_id, relative_path in sorted(curated_entries):
        entry = curated_entries[(root_id, relative_path)]
        path = entry["path"]
        index_aliases = sorted(entry["labels"], key=str.casefold)
        text = path.read_text(encoding="utf-8", errors="strict")
        meta, body = frontmatter(text)
        frontmatter_fallbacks += int(meta.get("_parse_status") == "scalar_fallback")
        content_hash = "sha256:" + sha256_bytes(path.read_bytes())
        direct, binding_state = local_lance_binding(meta, lance_bindings)
        if binding_state not in {"unbound", "matched"}:
            local_binding_skips[binding_state] += 1
            continue
        matches = normalized.get(sha256_text(norm(body)), [])
        local_anchor = local_nodes.get((root_id, path.name))
        if direct:
            record_id, merge_basis = direct, "explicit_lance_uuid"
        elif len(matches) == 1:
            record_id, merge_basis = matches[0], "exact_normalized_body"
        else:
            record_id, merge_basis = f"md:{root_id}:{relative_path}", "native_source"
        excerpt, excerpt_status = safe_excerpt(meta)
        aliases = alias_list(meta, path, index_aliases)
        source = {"root_id": root_id, "relative_path": relative_path, "source_hash": content_hash, "merge_basis": merge_basis}
        if record_id in records:
            merged += 1
            record = records[record_id]
            record["sources"].append(source)
            record["aliases"].extend(aliases)
            record["search_text"] += "\n" + "\n".join((str(meta.get("name") or ""), str(meta.get("description") or ""), body))
            record["project_id"] = record["project_id"] or project_id(meta, path)
            record["entity"] = record["entity"] or meta.get("entity")
            record["status"] = str(meta.get("status") or record["status"])
            record["supersedes"] = sorted(set(record["supersedes"] + string_list(meta.get("supersedes"))))
            if meta.get("superseded_by"):
                record["superseded_by_ref"] = meta.get("superseded_by")
            if excerpt_status == "accepted":
                record["safe_excerpt"], record["safe_excerpt_status"] = excerpt, excerpt_status
            continue
        kind = str(meta.get("type") or meta.get("kind") or "memory").lower()
        pid = project_id(meta, path)
        records[record_id] = {
            "record_id": record_id, "origin": "curated_md", "kind": kind, "project_id": pid, "entity": meta.get("entity"),
            "authority": str(meta.get("authority") or root_id), "authority_kind": "project_card" if kind == "project" else "curated_memory",
            "authority_rank": 60 if kind == "project" else 30, "authority_resolution": "resolved",
            "status": str(meta.get("status") or "active"), "valid_from": meta.get("valid_from"), "valid_to": meta.get("valid_to"),
            "recorded_time": file_iso(path), "supersedes": sorted(set(string_list(meta.get("supersedes")))),
            "superseded_by_ref": meta.get("superseded_by"),
            "superseded_by": None, "canonical_record_id": record_id, "content_hash": content_hash,
            "answerability": str(meta.get("answerability") or ("route_authority" if kind == "project" else "memory_allowed")),
            "safe_excerpt": excerpt, "safe_excerpt_status": excerpt_status, "source_pointer": f"{root_id}:{source['relative_path']}",
            "bti_anchor": local_anchor, "authority_verified_at": file_iso(path) if kind == "project" else None,
            "fresh_until": (datetime.fromtimestamp(path.stat().st_mtime, timezone.utc) + timedelta(days=14)).replace(microsecond=0).isoformat().replace("+00:00", "Z") if kind == "project" else None,
            "search_text": "\n".join((str(meta.get("name") or ""), str(meta.get("description") or ""), body)).strip(),
            "importance": float(meta.get("importance") or 0.7), "scope": str(meta.get("scope") or "global"),
            "active": 1, "compiled_at": file_iso(path), "aliases": aliases, "sources": [source],
        }

    for record in list(records.values()):
        if record.get("superseded_by_ref"):
            successor_id, kind = classify_record_ref(record["superseded_by_ref"], records, lance_bindings)
            if kind == "resolved" and successor_id != record["record_id"]:
                records[successor_id]["supersedes"].append(record["record_id"])
            else:
                lineage["skipped_superseded_by_" + ("self" if kind == "resolved" else kind)] += 1
    s2_conflicted = merge_supersession_lineage(records, all_lance_rows, lance_bindings, lineage)
    successors = defaultdict(set)
    for record in records.values():
        record["record_ref"] = make_ref("record", [record["record_id"]])
        resolved = set()
        for value in record["supersedes"]:
            predecessor, kind = classify_record_ref(value, records, lance_bindings)
            if kind == "resolved" and predecessor != record["record_id"]:
                resolved.add(predecessor)
            else:
                lineage["skipped_supersedes_" + ("self" if kind == "resolved" else kind)] += 1
        record["supersedes"] = sorted(resolved)
        record.pop("superseded_by_ref", None)
        record["aliases"] = list({canonical_json(item): item for item in record["aliases"]}.values())
        record["aliases"].sort(key=lambda item: (item["alias_norm"], item["alias"]))
        record["sources"] = list({canonical_json(item): item for item in record["sources"]}.values())
        record["sources"].sort(key=lambda item: (item["root_id"], item["relative_path"]))
        for predecessor in record["supersedes"]:
            successors[predecessor].add(record["record_id"])
    assert len({record["record_ref"] for record in records.values()}) == len(records), "record_ref collision"
    # A contradictory cycle is not resolved by order: its edges are dropped so publication continues, and
    # every member stays non-current with no successor chosen.
    cyclic = sorted((predecessor, next_id) for predecessor, nexts in successors.items() for next_id in nexts
                    if lineage_reaches(successors, next_id, predecessor))
    for predecessor, next_id in cyclic:
        successors[predecessor].discard(next_id)
        records[next_id]["supersedes"].remove(predecessor)
    for member in sorted({node for edge in cyclic for node in edge}):
        records[member]["status"] = "superseded"
        lineage["cycle_conflict_records"] += 1
    for predecessor, nexts in successors.items():
        if nexts:
            records[predecessor]["status"] = "superseded"
            if len(nexts) == 1 and predecessor not in s2_conflicted:
                records[predecessor]["superseded_by"] = records[predecessor]["canonical_record_id"] = next(iter(nexts))
            elif len(nexts) > 1:
                # Several live successors: the predecessor is superseded, but none of them is chosen as the true one.
                lineage["conflicting_successors"] += 1

    stats = {
        "lancedb_rows": len(lance_rows),
        "lancedb_rows_total": len(all_lance_rows),
        "lancedb_tombstones": lance_visibility.get("tombstoned", 0),
        "lancedb_visibility": dict(sorted(lance_visibility.items())),
        "local_lance_binding_skips": dict(sorted(local_binding_skips.items())),
        "supersession": dict(sorted(lineage.items())),
        "compiled_records": len(records),
        "indexed_md_files": indexed_counts,
        "memory_index_contract": index_ref_audit,
        "curated_md_merged": merged,
        "frontmatter_scalar_fallbacks": frontmatter_fallbacks,
        "bti_overlay": bti_stats,
        "raw_trace_included": False,
    }
    return sorted(records.values(), key=lambda item: item["record_id"]), stats


def compile_authority(records: list[dict]) -> list[dict]:
    config = json.loads(AUTHORITY_MAP_PATH.read_text(encoding="utf-8"))
    routes = []
    for route in config["routes"]:
        configured_resolution = route.get("resolution", "unresolved")
        if configured_resolution == "unresolved":
            root_id, relative, section = "", "", ""
            content_hash = route.get("expected_content_hash", "")
            authority_target = f"unresolved:{route['route_id']}"
            effective_resolution, fresh_until = "unresolved", None
        else:
            root_id = route["root_id"]
            base = AUTHORITY_ROOTS[root_id].resolve()
            relative = route["relative_path"].replace("\\", "/")
            section = route["section"]
            target = (base / relative).resolve()
            target.relative_to(base)
            text = target.read_text(encoding="utf-8", errors="strict")
            content_hash = "sha256:" + sha256_bytes(target.read_bytes())
            expected_hash = route.get("expected_content_hash")
            effective_resolution = configured_resolution
            if section not in text:
                effective_resolution = "section_missing"
            elif configured_resolution == "resolved" and expected_hash and content_hash != expected_hash:
                effective_resolution = "hash_mismatch"
            authority_target = f"authority:{root_id}:{relative}#{section}"
            fresh_until = route.get("fresh_until", config.get("default_fresh_until")) if effective_resolution == "resolved" else None
        target_ref = make_ref("authority", [root_id, relative, section]) if effective_resolution == "resolved" else None
        for alias in route["aliases"]:
            routes.append({
                "route_id": f"{route['route_id']}:{sha256_text(norm(alias))[:12]}", "alias_norm": norm(alias),
                "project_id": route.get("project_id"), "intent": route.get("intent", "status"),
                "authority_target": authority_target, "target_ref": target_ref,
                "root_id": root_id, "relative_path": relative, "section": section,
                "content_hash": content_hash, "answerability": route.get("answerability", "route_authority"),
                "resolution": effective_resolution, "fresh_until": fresh_until, "active": 1,
            })
    return sorted({item["route_id"]: item for item in routes}.values(), key=lambda item: item["route_id"])


def compile_entities() -> list[dict]:
    payload = json.loads(ENTITY_PATH.read_text(encoding="utf-8"))
    output = {}
    file_cache = {}

    def route_meta(pointer: str) -> dict | None:
        if pointer in file_cache:
            return file_cache[pointer]
        root_id, relative = pointer.split(":", 1)
        base = AUTHORITY_ROOTS.get(root_id)
        if base is None:
            file_cache[pointer] = None
            return None
        target = (base.resolve() / relative).resolve()
        try:
            safe_relative = target.relative_to(base.resolve()).as_posix()
        except ValueError:
            relocated = next(((candidate_id, target.relative_to(candidate_root.resolve()).as_posix()) for candidate_id, candidate_root in AUTHORITY_ROOTS.items() if target.is_relative_to(candidate_root.resolve())), None)
            if not relocated:
                file_cache[pointer] = None
                return None
            root_id, safe_relative = relocated
        if not target.is_file():
            file_cache[pointer] = None
            return None
        meta = {"root_id": root_id, "relative_path": safe_relative, "content_hash": "sha256:" + sha256_bytes(target.read_bytes())}
        file_cache[pointer] = meta
        return meta

    bridge_pointers = sorted({pointer for spec in payload.values() if isinstance(spec, dict) for candidate in [item for item in spec.get("candidates", []) if isinstance(item, dict)][:3] if (pointer := logical_pointer(str(candidate.get("path") or "")))})
    with ThreadPoolExecutor(max_workers=8, thread_name_prefix="mqe-hash") as pool:
        list(pool.map(route_meta, bridge_pointers, chunksize=64))
    for entity, spec in payload.items():
        if not isinstance(spec, dict):
            continue
        aliases = [entity, *spec.get("aliases", [])][:16]
        for candidate in [item for item in spec.get("candidates", []) if isinstance(item, dict)][:3]:
            pointer = logical_pointer(str(candidate.get("path") or ""))
            if not pointer:
                continue
            meta = route_meta(pointer)
            if not meta:
                continue
            for alias in aliases:
                alias_norm = norm(alias)
                if not 1 <= len(alias_norm) <= 160:
                    continue
                item = {
                    "alias_norm": alias_norm, "entity": str(entity)[:300], "source_pointer": pointer,
                    "entity_ref": make_ref("entity", [str(entity)[:300], pointer]), **meta,
                    "answerability": "pointer_only", "resolution": "resolved", "active": 1,
                    "score": float(candidate.get("score") or 0.5), "source": str(candidate.get("source") or "entity_bridge")[:80],
                }
                output[(alias_norm, item["entity"], pointer)] = item
    seeds = json.loads(ENTITY_ALIAS_PATH.read_text(encoding="utf-8"))
    assert seeds.get("schema_version") == "1.0.0"
    for entry in seeds.get("aliases", []):
        root_id = str(entry.get("root_id") or "")
        assert root_id in AUTHORITY_ROOTS, root_id
        base = AUTHORITY_ROOTS[root_id]
        relative = str(entry.get("relative_path") or "").replace("\\", "/")
        relative_path = Path(relative)
        assert not relative_path.is_absolute() and ".." not in relative_path.parts, relative
        target = base / relative_path
        assert target.is_file(), target
        alias_norm = norm(entry.get("alias"))
        assert 1 <= len(alias_norm) <= 2, alias_norm
        pointer = f"{root_id}:{relative}"
        meta = route_meta(pointer)
        assert meta, pointer
        item = {
            "alias_norm": alias_norm, "entity": str(entry.get("entity") or entry.get("alias"))[:300],
            "source_pointer": pointer, "entity_ref": make_ref("entity", [str(entry.get("entity") or entry.get("alias"))[:300], pointer]), **meta,
            "answerability": "pointer_only", "resolution": "resolved", "active": 1,
            "score": float(entry.get("score") or 1.0),
            "source": str(entry.get("source") or "curated_short_alias")[:80],
        }
        assert 0 < item["score"] <= 1.0
        output[(alias_norm, item["entity"], item["source_pointer"])] = item
    result = [output[key] for key in sorted(output)]
    assert all(item["entity_ref"].startswith("er1_") for item in result)
    return result


def connect(path: Path, initialize=False) -> sqlite3.Connection:
    db = sqlite3.connect(path, timeout=0.25)
    db.row_factory = sqlite3.Row
    db.execute("PRAGMA foreign_keys=ON")
    db.execute("PRAGMA busy_timeout=250")
    if initialize:
        db.executescript(SCHEMA_PATH.read_text(encoding="utf-8"))
    return db


def payload(record: dict) -> dict:
    return {"record": {key: value for key, value in record.items() if key not in {"aliases", "sources"}}, "aliases": record["aliases"], "sources": record["sources"]}


def queue(db: sqlite3.Connection, records: list[dict]) -> None:
    desired = {f"record:{item['record_id']}": payload(item) for item in records}
    previous = {row["source_key"]: dict(row) for row in db.execute("SELECT source_key,source_hash,record_id FROM source_state")}
    with db:
        for source_key, body in desired.items():
            source_hash = "sha256:" + sha256_text(canonical_json(body))
            if source_key in previous and previous[source_key]["source_hash"] == source_hash:
                continue
            event_id = "evt:" + sha256_text(f"{source_key}|upsert|{source_hash}")
            # One live event per source: a superseded pending event is dropped, and an event that was applied before is
            # re-armed so a payload that returns to an earlier state (A->B->A) is applied again; a pending one is kept as is.
            db.execute("DELETE FROM outbox WHERE source_key=? AND state='pending' AND event_id<>?", (source_key, event_id))
            db.execute("INSERT INTO outbox(event_id,source_key,operation,payload_json,state,created_at) VALUES(?,?,'upsert',?,'pending',?) ON CONFLICT(event_id) DO UPDATE SET state='pending',payload_json=excluded.payload_json,created_at=excluded.created_at,applied_at=NULL,error=NULL WHERE outbox.state<>'pending'", (event_id, source_key, canonical_json(body), NOW_ISO))
            db.execute("INSERT INTO source_state(source_key,source_hash,record_id,observed_at) VALUES(?,?,?,?) ON CONFLICT(source_key) DO UPDATE SET source_hash=excluded.source_hash,record_id=excluded.record_id,observed_at=excluded.observed_at", (source_key, source_hash, body["record"]["record_id"], NOW_ISO))
        for source_key, old in previous.items():
            if source_key in desired:
                continue
            event_id = "evt:" + sha256_text(f"{source_key}|delete|{old['source_hash']}")
            db.execute("DELETE FROM outbox WHERE source_key=? AND state='pending' AND event_id<>?", (source_key, event_id))
            db.execute("INSERT INTO outbox(event_id,source_key,operation,payload_json,state,created_at) VALUES(?,?,'delete',?,'pending',?) ON CONFLICT(event_id) DO UPDATE SET state='pending',payload_json=excluded.payload_json,created_at=excluded.created_at,applied_at=NULL,error=NULL WHERE outbox.state<>'pending'", (event_id, source_key, canonical_json({"record_id": old["record_id"]}), NOW_ISO))
            db.execute("DELETE FROM source_state WHERE source_key=?", (source_key,))


def apply_pending(db: sqlite3.Connection) -> int:
    events = list(db.execute("SELECT * FROM outbox WHERE state='pending' ORDER BY created_at,event_id"))
    columns = ("record_id","record_ref","origin","kind","project_id","entity","authority","authority_kind","authority_rank","authority_resolution","status","valid_from","valid_to","recorded_time","supersedes","superseded_by","canonical_record_id","content_hash","answerability","safe_excerpt","safe_excerpt_status","source_pointer","bti_anchor","authority_verified_at","fresh_until","search_text","importance","scope","active","compiled_at")
    inserts = ",".join("?" for _ in columns)
    updates = ",".join(f"{name}=excluded.{name}" for name in columns if name != "record_id")
    with db:
        for event in events:
            db.execute("SAVEPOINT apply_event")
            try:
                body = json.loads(event["payload_json"])
                if event["operation"] == "delete":
                    record_id = body["record_id"]
                    db.execute("DELETE FROM records_fts WHERE record_id=?", (record_id,))
                    db.execute("DELETE FROM records WHERE record_id=?", (record_id,))
                else:
                    record = body["record"]
                    record["supersedes"] = canonical_json(record.get("supersedes", []))
                    db.execute(f"INSERT INTO records({','.join(columns)}) VALUES({inserts}) ON CONFLICT(record_id) DO UPDATE SET {updates}", tuple(record.get(name) for name in columns))
                    record_id = record["record_id"]
                    for table in ("aliases", "short_grams", "record_sources"):
                        db.execute(f"DELETE FROM {table} WHERE record_id=?", (record_id,))
                    db.execute("DELETE FROM supersession_edges WHERE successor_id=?", (record_id,))
                    db.execute("DELETE FROM records_fts WHERE record_id=?", (record_id,))
                    alias_text = []
                    for alias in body["aliases"]:
                        db.execute("INSERT OR IGNORE INTO aliases(alias_norm,record_id,alias,source,confidence) VALUES(?,?,?,?,?)", (alias["alias_norm"], record_id, alias["alias"], alias["source"], alias["confidence"]))
                        alias_text.append(alias["alias"])
                        gram = loose(alias["alias"])
                        if len(gram) in (1, 2) and alias["source"] == "frontmatter:aliases":
                            db.execute("INSERT OR IGNORE INTO short_grams(gram,record_id,source,confidence) VALUES(?,?,?,?)", (gram, record_id, alias["source"], alias["confidence"]))
                    for source in body["sources"]:
                        db.execute("INSERT OR IGNORE INTO record_sources(record_id,root_id,relative_path,source_hash,merge_basis) VALUES(?,?,?,?,?)", (record_id, source["root_id"], source["relative_path"], source["source_hash"], source["merge_basis"]))
                    for predecessor in json.loads(record["supersedes"]):
                        db.execute("INSERT OR IGNORE INTO supersession_edges(predecessor_id,successor_id) VALUES(?,?)", (predecessor, record_id))
                    db.execute("INSERT INTO records_fts(record_id,search_text) VALUES(?,?)", (record_id, "\n".join((*alias_text, record["search_text"]))))
                compact_payload = canonical_json({"record_id": record_id, "source_key": event["source_key"]})
                db.execute("UPDATE outbox SET state='applied',payload_json=?,applied_at=?,error=NULL WHERE event_id=?", (compact_payload, NOW_ISO, event["event_id"]))
                db.execute("RELEASE SAVEPOINT apply_event")
            except Exception as error:
                db.execute("ROLLBACK TO SAVEPOINT apply_event")
                db.execute("RELEASE SAVEPOINT apply_event")
                db.execute("UPDATE outbox SET state='failed',applied_at=?,error=? WHERE event_id=?", (NOW_ISO, str(error)[:500], event["event_id"]))
                db.commit()
                raise
        # A successor that left the projection keeps its old edge. Drop such an edge only when its predecessor left too or is
        # still published as superseded (e.g. retired by a forgotten successor's key), so the removal never revives it; the
        # active predecessor of a legacy card-only chain keeps the edge (decided).
        db.execute("DELETE FROM supersession_edges WHERE successor_id NOT IN (SELECT record_id FROM records) AND (predecessor_id NOT IN (SELECT record_id FROM records) OR predecessor_id IN (SELECT record_id FROM records WHERE lower(status)='superseded'))")
    return len(events)


def replace_routes(db: sqlite3.Connection, authorities: list[dict], entities: list[dict]) -> None:
    with db:
        db.execute("DELETE FROM authority_routes")
        db.execute("DELETE FROM entity_routes")
        db.executemany("INSERT INTO authority_routes(route_id,alias_norm,project_id,intent,authority_target,target_ref,root_id,relative_path,section,content_hash,answerability,resolution,fresh_until,active) VALUES(:route_id,:alias_norm,:project_id,:intent,:authority_target,:target_ref,:root_id,:relative_path,:section,:content_hash,:answerability,:resolution,:fresh_until,:active)", authorities)
        db.executemany("INSERT OR IGNORE INTO entity_routes(alias_norm,entity,source_pointer,entity_ref,root_id,relative_path,content_hash,answerability,resolution,active,score,source) VALUES(:alias_norm,:entity,:source_pointer,:entity_ref,:root_id,:relative_path,:content_hash,:answerability,:resolution,:active,:score,:source)", entities)


def checksum(db: sqlite3.Connection) -> str:
    selects = (
        "SELECT record_id,record_ref,origin,kind,project_id,entity,authority,authority_kind,authority_rank,authority_resolution,status,valid_from,valid_to,recorded_time,supersedes,superseded_by,canonical_record_id,content_hash,answerability,safe_excerpt,safe_excerpt_status,source_pointer,bti_anchor,authority_verified_at,fresh_until,search_text,printf('%.17g',importance) AS importance,scope,active FROM records ORDER BY record_id",
        "SELECT alias_norm,record_id,alias,source,printf('%.17g',confidence) AS confidence FROM aliases ORDER BY alias_norm,record_id,alias",
        "SELECT gram,record_id,source,printf('%.17g',confidence) AS confidence FROM short_grams ORDER BY gram,record_id,source",
        "SELECT record_id,root_id,relative_path,source_hash,merge_basis FROM record_sources ORDER BY record_id,root_id,relative_path",
        "SELECT route_id,alias_norm,project_id,intent,authority_target,target_ref,root_id,relative_path,section,content_hash,answerability,resolution,fresh_until,active FROM authority_routes ORDER BY route_id",
        "SELECT alias_norm,entity,source_pointer,entity_ref,root_id,relative_path,content_hash,answerability,resolution,active,printf('%.17g',score) AS score,source FROM entity_routes ORDER BY alias_norm,entity,source_pointer",
        "SELECT predecessor_id,successor_id FROM supersession_edges ORDER BY predecessor_id,successor_id",
        "SELECT record_id,search_text FROM records_fts ORDER BY record_id,search_text",
    )
    rows = [[dict(row) for row in db.execute(sql)] for sql in selects]
    contract = {"checksum_contract": CHECKSUM_CONTRACT, "schema_version": SCHEMA_VERSION, "ref_key_id": REF_KEY_ID or db.execute("SELECT value FROM projection_meta WHERE key='ref_key_id'").fetchone()[0], "schema_sha256": sha256_bytes(SCHEMA_PATH.read_bytes()), "logical_tables": rows}
    return "sha256:" + sha256_text(canonical_json(contract))


def validate(db: sqlite3.Connection) -> dict:
    assert db.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
    assert db.execute("SELECT COUNT(*) FROM outbox WHERE state='failed'").fetchone()[0] == 0
    assert db.execute("PRAGMA user_version").fetchone()[0] == 2
    assert db.execute("SELECT COUNT(*) FROM records WHERE record_ref IS NULL OR length(record_ref)<>68").fetchone()[0] == 0
    assert db.execute("SELECT COUNT(*) FROM (SELECT record_ref FROM records GROUP BY record_ref HAVING COUNT(*)>1)").fetchone()[0] == 0
    assert db.execute("SELECT COUNT(*) FROM authority_routes WHERE resolution='resolved' AND target_ref IS NULL").fetchone()[0] == 0
    assert db.execute("SELECT COUNT(*) FROM entity_routes WHERE entity_ref IS NULL OR resolution<>'resolved' OR active<>1").fetchone()[0] == 0
    assert db.execute("SELECT COUNT(*) FROM records WHERE lower(kind) IN ('raw_event','raw_trace','session_trace')").fetchone()[0] == 0
    fence = chr(96) * 3
    assert db.execute("SELECT COUNT(*) FROM records WHERE safe_excerpt_status='accepted' AND (lower(safe_excerpt) LIKE '%[ai-relay]%' OR lower(safe_excerpt) LIKE '%ignore previous%' OR instr(safe_excerpt,?)>0)", (fence,)).fetchone()[0] == 0
    cycle = db.execute("WITH RECURSIVE walk(start,node) AS (SELECT predecessor_id,successor_id FROM supersession_edges UNION SELECT walk.start,e.successor_id FROM walk JOIN supersession_edges e ON e.predecessor_id=walk.node) SELECT COUNT(*) FROM walk WHERE start=node").fetchone()[0]
    assert cycle == 0
    return {
        "integrity_check": "ok",
        "record_count": db.execute("SELECT COUNT(*) FROM records").fetchone()[0],
        "fts_count": db.execute("SELECT COUNT(*) FROM records_fts").fetchone()[0],
        "alias_count": db.execute("SELECT COUNT(*) FROM aliases").fetchone()[0],
        "authority_route_count": db.execute("SELECT COUNT(*) FROM authority_routes").fetchone()[0],
        "entity_route_count": db.execute("SELECT COUNT(*) FROM entity_routes").fetchone()[0],
        "raw_trace_rows": 0,
        "unsafe_excerpt_rows": 0,
        "supersession_cycles": cycle,
    }


run_id = "run:" + uuid.uuid4().hex
start_current_bytes = CURRENT_PATH.read_bytes() if CURRENT_PATH.is_file() else None
if EXPECTED_CURRENT_SHA256 is not None:
    assert start_current_bytes is not None
    assert sha256_bytes(start_current_bytes) == EXPECTED_CURRENT_SHA256, "projection_publication_base_changed"

if MODE == "verify":
    pointer = json.loads(CURRENT_PATH.read_text(encoding="utf-8"))
    assert set(pointer) == {"checksum", "corpus_epoch", "filename", "ref_key_id", "schema_version", "updated_at"}
    db_path = DATA_DIR / pointer["filename"]
    db = connect(db_path)
    observed = checksum(db)
    meta = dict(db.execute("SELECT key,value FROM projection_meta"))
    epoch = observed.removeprefix("sha256:")[:16]
    assert pointer["schema_version"] == meta.get("schema_version") == SCHEMA_VERSION
    assert pointer["ref_key_id"] == meta.get("ref_key_id")
    assert re.fullmatch(r"rk1_[0-9a-f]{32}", pointer["ref_key_id"])
    assert meta.get("checksum_contract") == CHECKSUM_CONTRACT
    assert observed == pointer["checksum"] == meta.get("checksum")
    assert pointer["corpus_epoch"] == meta.get("corpus_epoch") == epoch
    assert pointer["filename"] == f"memory_index.{epoch}.sqlite"
    result = {"mode": MODE, "database": db_path.name, "checksum": observed, "validation": validate(db)}
    db.close()
else:
    if start_current_bytes is not None:
        current_pointer = json.loads(start_current_bytes.decode("utf-8"))
        if current_pointer.get("schema_version") == SCHEMA_VERSION and current_pointer.get("ref_key_id") != REF_KEY_ID:
            assert MODE == "rebuild" and os.environ.get("MQE_ALLOW_REF_KEY_ROTATION") == "1", "ref key rotation requires explicit full rebuild"
        if MODE == "repair":
            assert current_pointer.get("schema_version") == SCHEMA_VERSION and current_pointer.get("ref_key_id") == REF_KEY_ID, "repair requires current v2 ref key"
    records, source_stats = compile_records()
    authorities = compile_authority(records)
    entities = compile_entities()
    if MODE in {"rebuild", "repair"}:
        db_path = DATA_DIR / f".staging-{uuid.uuid4().hex}.sqlite"
        db = connect(db_path, initialize=True)
    else:
        assert start_current_bytes is not None, "incremental projection requires CURRENT"
        pointer = json.loads(start_current_bytes.decode("utf-8"))
        source_path = DATA_DIR / pointer["filename"]
        source_db = connect(source_path)
        assert source_db.execute("PRAGMA user_version").fetchone()[0] == 2, "incremental requires v2 projection"
        source_meta = dict(source_db.execute("SELECT key,value FROM projection_meta"))
        assert source_meta.get("schema_version") == SCHEMA_VERSION and source_meta.get("ref_key_id") == REF_KEY_ID
        db_path = DATA_DIR / f".staging-{uuid.uuid4().hex}.sqlite"
        db = connect(db_path)
        source_db.backup(db)
        source_db.close()
    db.execute("INSERT INTO projection_runs(run_id,mode,started_at,status) VALUES(?,?,?,'running')", (run_id, MODE, NOW_ISO))
    queue(db, records)
    applied = apply_pending(db)
    replace_routes(db, authorities, entities)
    observed = checksum(db)
    expected_logical_checksum = os.environ.get("MQE_EXPECTED_LOGICAL_CHECKSUM")
    if expected_logical_checksum is not None:
        assert re.fullmatch(r"sha256:[0-9a-f]{64}", expected_logical_checksum), "projection_expected_checksum_invalid"
        assert observed == expected_logical_checksum, "projection_expected_checksum_mismatch"
    epoch = observed.removeprefix("sha256:")[:16]
    validation = validate(db)
    with db:
        meta = {"schema_version": SCHEMA_VERSION, "checksum_contract": CHECKSUM_CONTRACT, "ref_key_id": REF_KEY_ID, "corpus_epoch": epoch, "checksum": observed, "raw_trace_included": "0", "built_at": NOW_ISO, "mode": MODE}
        db.executemany("INSERT INTO projection_meta(key,value) VALUES(?,?) ON CONFLICT(key) DO UPDATE SET value=excluded.value", sorted(meta.items()))
        db.execute("UPDATE projection_runs SET finished_at=?,source_count=?,record_count=?,checksum=?,status='complete' WHERE run_id=?", (NOW_ISO, source_stats["lancedb_rows"] + sum(source_stats["indexed_md_files"].values()), len(records), observed, run_id))
    db.execute("PRAGMA wal_checkpoint(TRUNCATE)")
    db.close()
    if MODE in {"rebuild", "repair", "incremental"}:
        final_path = DATA_DIR / f"memory_index.{epoch}.sqlite"
        try:
            os.link(db_path, final_path)
        except FileExistsError:
            check_db = connect(final_path)
            existing_checksum = checksum(check_db)
            check_db.close()
            if existing_checksum != observed:
                raise AssertionError((existing_checksum, observed))
        db_path.unlink()
        db_path = final_path
    pointer = {"schema_version": SCHEMA_VERSION, "ref_key_id": REF_KEY_ID, "corpus_epoch": epoch, "filename": db_path.name, "checksum": observed, "updated_at": PUBLICATION_UPDATED_AT}
    result = {
        "schema_version": SCHEMA_VERSION, "ref_key_id": REF_KEY_ID, "mode": MODE, "built_at": NOW_ISO, "database": db_path.name,
        "checksum": observed, "corpus_epoch": epoch, "source_stats": source_stats,
        "authority_routes": len(authorities), "entity_routes": len(entities),
        "applied_outbox_events": applied, "validation": validation, "raw_trace_included": False,
        "publication_updated_at": PUBLICATION_UPDATED_AT,
    }
    pointer_bytes = (json.dumps(pointer, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
    manifest_bytes = (json.dumps(result, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
    pointer_tmp = DATA_DIR / f".CURRENT.{uuid.uuid4().hex}.tmp"
    manifest_tmp = DATA_DIR / f".projection_manifest.{uuid.uuid4().hex}.tmp"
    publication_db = sqlite3.connect(PUBLICATION_LOCK_PATH, timeout=60.0, isolation_level=None)
    try:
        publication_db.execute("BEGIN IMMEDIATE")
        live_current_bytes = CURRENT_PATH.read_bytes() if CURRENT_PATH.is_file() else None
        assert live_current_bytes == start_current_bytes, "projection_publication_base_changed"
        for temporary, payload in ((manifest_tmp, manifest_bytes), (pointer_tmp, pointer_bytes)):
            with temporary.open("xb") as handle:
                handle.write(payload)
                handle.flush()
                os.fsync(handle.fileno())
        os.replace(pointer_tmp, CURRENT_PATH)
        os.replace(manifest_tmp, MANIFEST_PATH)
        publication_db.execute("COMMIT")
    except BaseException:
        try:
            publication_db.execute("ROLLBACK")
        except sqlite3.Error:
            pass
        raise
    finally:
        publication_db.close()
        pointer_tmp.unlink(missing_ok=True)
        manifest_tmp.unlink(missing_ok=True)

print(json.dumps(result, ensure_ascii=False, indent=2))
